# 1.介绍

assignment1 要求手动实现：
1. BPE
2. LLM（gpt 的decoder-only）
3. cross-entropy loss & AdamW optimizer
4. train

# 2.实现

先说一下实现思路，其实 adapters.py 中将所有需要实现的接口都留出来了，这里以 softmax 为例：
```python
def run_softmax(in_features: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    """
    Given a tensor of inputs, return the output of softmaxing the given `dim`
    of the input.

    Args:
        in_features (Float[Tensor, "..."]): Input features to softmax. Shape is arbitrary.
        dim (int): Dimension of the `in_features` to apply softmax to.

    Returns:
        Float[Tensor, "..."]: Tensor of with the same shape as `in_features` with the output of
        softmax normalizing the specified `dim`.
    """
    raise NotImplementedError
```
介绍了传入的参数是什么样，你应该实现什么，所以只需要按照 adapters.py 中的接口进行实现，然后使用 uv 进行测试即可

In [2]:
from __future__ import annotations
from typing import Iterable
import torch
from torch import Tensor
import torch.nn.functional as F  # 只用来对齐参考时自测，不在最终实现中依赖
import math
from typing import Iterable, Optional
from torch.optim import Optimizer
import json
import os
import regex as rem
from collections.abc import Iterable, Iterator
from jaxtyping import Bool, Float, Int
from einops import einsum
import torch.nn as nn

## 2.1 工程逻辑

其实把所有的实现都写到 adapters.py 中即可，但是既然都学 Language Modeling from Scratch 了，那就按一个完整的工程来进行工作。

下面是我设计的工程目录：
```
tests/
└─ adapters.py             # 仅定义 run_*，内部调用 src/* 的实现
src/
├─ nn_utils.py             # softmax / cross_entropy / gradient_clipping
├─ optim_sched.py          # AdamW 类选择 / 余弦+warmup LR
├─ data.py                 # get_batch
├─ io.py                   # save/load checkpoint
├─ bpe/
│   ├─ tokenizer.py        # get_tokenizer
│   └─ train_bpe.py        # run_train_bpe
└─ model/
   ├─ attention.py        # sdpa / mha / rope / mha_with_rope
   ├─ layers.py           # linear / embedding / swiglu / rmsnorm
   └─ transformer.py      # block / lm
```

# 3 nn_utils

这一部分就是实现 torch 中的各种工具，其中包括：
1. Softmax
2. cross_entropy
3. gradient_clipping
4. silu

## 3.1 Softmax

Softmax：多分类问题的输出层，将一组任意实数转换为一个概率分布


数学定义：假设我们有一个包含K个实数的向量 **z** = ($z_1$, $z_2$, ..., $z_K$)，Softmax函数会计算一个新的向量 **σ(z)** = ($\sigma_1$, $\sigma_2$, ..., $\sigma_K$)，其中每个元素 $\sigma_i$ 的计算公式如下：

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} \quad \text{for } i = 1, \dots, K$$

在实际 PyTorch 实现中 Softmax 实现有多种优化。这里我们只说数值稳定性优化：“减去最大值技巧 (Subtract-Max Trick)”，这是Softmax实现中**最重要**的优化，用于防止计算过程中的数值上溢（overflow）和下溢（underflow）。

Softmax的核心计算是 $e^{z\_i}$。如果输入的logits向量 `z` 中包含较大的数值（例如，`z_i = 1000`），$e^{1000}$ 的结果会是一个巨大的数字，超出浮点数能表示的范围，导致**上溢 (Overflow)**，结果变为 `inf`。这会导致最终的概率分布变成 `[nan, nan, ...]`。

反之，如果logits都为非常小的负数（例如，`z_i = -1000`），$e^{-1000}$ 的结果会无限接近于0，导致**下溢 (Underflow)**。如果分子和分母都下溢为0，最终结果也会是 `nan`。

PyTorch 给输入向量的所有元素加上或减去同一个常数，其输出结果保持不变。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} = \frac{C \cdot e^{z_i}}{C \cdot \sum_{j=1}^{K} e^{z_j}} = \frac{e^{z_i + \log(C)}}{\sum_{j=1}^{K} e^{z_j + \log(C)}}$$

我们可以选择一个特定的常数 `C` 来优化计算。最佳选择是令 $\\log(C) = - \\max(\\mathbf{z})$，即从所有logits中减去它们的最大值。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j=1}^{K} e^{z_j - \max(\mathbf{z})}}$$

**这样做的好处是：**

1.  **防止上溢**：变换后的向量中，最大的元素是 `0` ($e^0=1$)，所有其他元素都是负数。这样就保证了指数计算的最大结果是 `1`，有效避免了上溢。
2.  **减少下溢风险**：通过将整个数值范围向上“平移”，至少保证了有一个元素（即原始最大值对应的元素）的指数结果为 `1`，使得分母至少为 `1`，从而避免了分母因所有项都下溢成零而导致的除零错误。


In [3]:
def softmax(in_features: Tensor, dim: int) -> Tensor:
    """
    数值稳定 softmax：对 in_features 沿 dim 做 softmax，输出形状与输入一致。
    关键：先减去该维最大值，避免 exp 溢出。
    """
    shifted = in_features - in_features.max(dim=dim, keepdim=True).values
    exps = torch.exp(shifted)
    return exps / exps.sum(dim=dim, keepdim=True)

## 3.2 Cross_entropy

在信息论中，**熵 (Entropy)** 用来衡量一个概率分布的“不确定性”或“信息量”。一个系统越混乱、越不可预测，它的熵就越高。

  * **直观例子**：
      * **低熵**：一枚“作弊”的硬币，99%的概率是正面。结果非常确定，所以熵很低。
      * **高熵**：一枚均匀的硬币，正反面概率各50%。结果最不确定，所以熵最高。

而**交叉熵 (Cross-Entropy)** 则更进一步，它用来衡量**两个概率分布之间的差异**，一般用于衡量真实分布（ground truth）与模型预测分布之间的差距。差距越小，模型越好。具体来说，它衡量的是，当我们**使用一个“错误的”或“近似的”概率分布 `q` 来表示一个“真实的”概率分布 `p` 时，所需要付出的额外信息量（或编码长度）**。

如果近似分布 `q` 与真实分布 `p` 非常接近，那么交叉熵的值就很低。反之，如果 `q` 与 `p` 相差甚远，交叉熵的值就会很高。

**数学定义与公式**：

假设我们有两个离散的概率分布，$p$ 和 $q$。

  * $p$: 真实分布 (True Distribution)。一般为数据的真实标签，表示为 one-hot 编码。
  * $q$: 预测分布 (Predicted Distribution)。模型的输出，通常是经过 Softmax 或 Sigmoid 函数处理后的概率。

交叉熵 $H(p, q)$ 的计算公式如下：

$$H(p, q) = - \sum_{i=1}^{K} p(x_i) \log(q(x_i))$$

其中：

  * $K$ 是所有可能事件（类别）的总数。
  * $p(x_i)$ 是事件 $x_i$ 在真实分布 $p$ 中的概率。
  * $q(x_i)$ 是事件 $x_i$ 在预测分布 $q$ 中的概率。

**注意**：我们在进行计算的时候不用完全照搬公式来计算，因为真实分布 $p$ 是 one-hot 编码的，假设真实类别是 $c$，那么只有 $p(x_c)=1$，而所有其他的 $p(x_i)=0$ (当 $i \neq c$ 时)。这样一来，上面的求和公式就可以大大简化：

$$
\begin{aligned}
H(p, q) &= -\sum_{i=1}^{V} p(x_i) \log(q(x_i)) \\
        &= -(p(x_1)\log(q(x_1)) + \dots + p(x_c)\log(q(x_c)) + \dots + p(x_V)\log(q(x_V))) \\
        &= -(0 \cdot \log(q(x_1)) + \dots + 1 \cdot \log(q(x_c)) + \dots + 0 \cdot \log(q(x_V))) \\
        &= -\log(q(x_c))
\end{aligned}
$$

所以，交叉熵损失函数最终要计算的，就是**模型预测的正确类别所对应的概率的负对数值**。我们的目标就是让这个损失值越小越好，也就是让 $q_c$ (正确类别的概率) 越接近 1 越好。

---

**具体的计算步骤如下：**
1. 任务设定与记号

* 输入 `inputs ∈ ℝ^{B×V}`：每个样本的未归一化分数，`B` 批大小，`V` 类别/词表大小。第 `i` 个样本的向量记为 `z_i ∈ ℝ^V`。
* 标签 `targets ∈ {0,…,V-1}^B`：第 `i` 个样本的真实类别 `y_i`。

**softmax 概率：**
$$
q_i[c] \equiv \text{softmax}(z_i)[c] = \frac{e^{z_{i,c}}}{\sum_{j=1}^V e^{z_{i,j}}}
$$

**交叉熵（对 one-hot 真实分布 $p_i$）**：
$$
\mathrm{CE}(p_i,q_i) = -\sum_{c=1}^V p_i[c]\log q_i[c]
$$
当 $p_i$ 是 one-hot 且真实类为 $y_i$ 时，
$$
\mathrm{CE}(p_i,q_i) = -\log q_i[y_i]
$$

**按 batch 取平均：**
$$
\mathcal{L} = \frac1B \sum_{i=1}^B \big(-\log q_i[y_i]\big)
$$

2. 计算式

把 $-\log q_i[y_i]$ 展开：
$$
-\log\frac{e^{z_{i,y_i}}}{\sum_j e^{z_{i,j}}}
= -z_{i,y_i} + \log\sum_j e^{z_{i,j}}
$$

**数值稳定 trick：**
直接算 $\sum e^{z}$ 容易上溢。用
$$
\log\sum_j e^{z_j}
= m + \log\sum_j e^{z_j-m},\quad m=\max_j z_j
$$
> PyTorch 的 `torch.logsumexp` 已内置这个稳定化。

3. 代码讲解

```python
logsumexp = torch.logsumexp(inputs, dim=1, keepdim=True)   # (B,1)
```

对应 $\log\sum_j e^{z_{i,j}}$（对每个样本一行做）。

```python
log_probs = inputs - logsumexp                             # (B,V)
```

这就是 $\log \text{softmax}(z_i)$：
$$
\log q_i[c] = z_{i,c} - \log\sum_j e^{z_{i,j}}
$$

```python
gathered = log_probs.gather(1, targets.view(-1,1)).squeeze(1)  # (B,)
```

从每行挑出正确类别 $c=y_i$ 的 $\log q_i[y_i]$。

```python
return -gathered.mean()
```

做 $-\log q_i[y_i]$ 并对 batch 取均值，即
$$
\mathcal{L}=\frac1B\sum_i \left(-\log q_i[y_i]\right)
$$

In [4]:
def cross_entropy(inputs: Tensor, targets: Tensor) -> Tensor:
    """
    稳定的交叉熵：inputs 形状 (B, V) 为未归一化 logits；targets 形状 (B,) 为 Long 类别 id。
    等价于 F.cross_entropy(inputs, targets, reduction='mean')，但用 logsumexp 保证稳定性。
    """
    # log_softmax(x) = x - logsumexp(x)
    logsumexp = torch.logsumexp(inputs, dim=1, keepdim=True)   # (B, 1)
    log_probs = inputs - logsumexp                             # (B, V)
    gathered = log_probs.gather(1, targets.view(-1, 1)).squeeze(1)  # (B,)
    return -gathered.mean()                                    # 标准 mean reduction

## 3.3 梯度裁剪 (Gradient Clipping) 

梯度裁剪是一种简单而有效的技术，用来解决梯度爆炸问题。形象地来说就是给模型设定一个“最大步长”的安全限制。

**核心思想**：在更新模型参数之前，检查所有梯度的“总长度”（即范数 L2-Norm）。

  * **如果总长度超过了你设定的阈值**：就按比例**缩小所有梯度**，使得它们的总长度刚好等于这个阈值。重要的是，**所有梯度都被同一个系数缩小**，所以梯度的**方向保持不变**，只是步子的大小被限制了。
  * **如果总长度没有超过阈值**：那说明梯度是正常的，什么也不用做。

In [5]:
def gradient_clipping(parameters: Iterable[torch.nn.Parameter], max_l2_norm: float) -> None:
    """
    全局 L2 范数裁剪：只处理 p.grad 非空的参数。若全局范数超过阈值，则用同一个系数原地缩放每个 grad。
    与 torch.nn.utils.clip_grad.clip_grad_norm_ 的语义一致。
    """
    params = [p for p in parameters if p.grad is not None]
    if not params:
        return
    device = params[0].grad.device
    # 先算每个 grad 的 2-范数，再做“范数的范数”得到全局范数
    grads_norms = torch.stack([p.grad.detach().norm(2) for p in params]).to(device)
    total_norm = grads_norms.norm(2)
    clip_coef = max_l2_norm / (total_norm + 1e-6)  # +epsilon 防 0
    if clip_coef < 1.0:
        for p in params:
            p.grad.detach().mul_(clip_coef.to(p.grad.device))  # 原地缩放

## 3.4 Swish 激活函数：SiLU（Sigmoid Linear Unit）

**SiLU**的数学公式为：

$$
\text{SiLU}(x) = x \cdot \sigma(x)
$$

其中：

* $\sigma(x)$ 是 Sigmoid 函数：

  $$
  \sigma(x) = \frac{1}{1 + e^{-x}}
  $$
* 所以：

  $$
  \text{SiLU}(x) = \frac{x}{1 + e^{-x}}
  $$

**直观理解**

* **ReLU (Rectified Linear Unit)**:
  $\text{ReLU}(x) = \max(0, x)$，对负数直接裁掉。
* **Sigmoid**:
  把所有输入压缩到 $(0,1)$。
* **SiLU** 则是：在 Sigmoid 的“门控”作用下保留输入 $x$ 的一部分：

  * 当 $x$ 很大时，$\sigma(x) \approx 1$，所以 $\text{SiLU}(x) \approx x$。
  * 当 $x$ 很小时，$\sigma(x) \approx 0$，所以 $\text{SiLU}(x) \approx 0$。
  * 在负数区域，Sigmoid 还没完全关掉，输出是一个 **小的负值**，所以不像 ReLU 那样直接“截断”。

这就是为什么 SiLU 被称为 **平滑版 ReLU**。

**形状特性**

* 当 $x \to +\infty$，$\text{SiLU}(x) \approx x$。
* 当 $x \to -\infty$，$\text{SiLU}(x) \approx 0$（趋近于0，但略带负值）。
* 在 $x=0$ 附近，曲线平滑，避免了 ReLU 的 **不连续点**。
* 函数是 **非单调** 的：在 $x < 0$ 的一小段区间，曲线先下降一点点，再上升。


**导数推导**

在反向传播中，我们需要 $\frac{d}{dx}\text{SiLU}(x)$。

$$
\text{SiLU}(x) = x \cdot \sigma(x)
$$

利用积的导数法则：

$$
\frac{d}{dx} \text{SiLU}(x) = \sigma(x) + x \cdot \sigma'(x)
$$

而 Sigmoid 的导数：

$$
\sigma'(x) = \sigma(x)(1-\sigma(x))
$$

所以：

$$
\frac{d}{dx} \text{SiLU}(x) = \sigma(x) + x \cdot \sigma(x)(1-\sigma(x))
$$

进一步化简：

$$
\frac{d}{dx} \text{SiLU}(x) = \sigma(x)\big(1 + x(1-\sigma(x))\big)
$$

这说明它的梯度是平滑的，不会像 ReLU 那样在 0 点突然变成 0 或 1。

**为什么有用？**

1. **平滑性**：没有 ReLU 的“硬拐点”，优化更稳定。
2. **允许负值**：不像 ReLU 那样直接丢掉负数，保留了一些负信息。
3. **非单调性**：在小负区间会出现轻微下陷，被认为对深度模型的特征表达更有利。
4. **性能验证**：在 Google 的论文 *Swish: a Self-Gated Activation Function* 中，SiLU 在图像识别、机器翻译等任务上优于 ReLU。
   （很多模型默认用 SiLU，比如 **EfficientNet**, **YOLOv5/YOLOv8** 等）

In [6]:
def silu(x: Float[Tensor, "..."]) -> Float[Tensor, "..."]:
    return x * torch.sigmoid(x)

## 3.5 测试

使用 
```bash
uv run -- pytest -x -vv -k nn_utils
``` 
进行测试即可

# 4 optim_sched

这一部分是实现：
1. AdamW
2. 余弦学习率调度lr_cosine_schedule

## 4.1 AdamW
```python
from torch.optim import Optimizer
```

注解：
* `Optimizer`：PyTorch 的优化器基类，继承它可自动获得**参数组（param\_groups）**、**state 管理**等通用框架。

---


> 基础数学知识
> 
> 标准 Adam（Kingma & Ba, 2014）对梯度 $g_t$ 维护一阶、二阶动量：
>
> $$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat m_t &= \frac{m_t}{1-\beta_1^t},\quad
\hat v_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t - \alpha \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
\end{aligned}
$$
>
> 传统 “L2 正则” 往往体现在把 $\lambda\theta$ 加到梯度中（即**耦合式** weight decay）。
> **AdamW（Loshchilov & Hutter, 2017）**提出把 weight decay 从梯度里**拿出来单独对参数衰减**（**解耦**），更新式变为：
>
> $$
\theta_{t+1} = \underbrace{\theta_t - \alpha\lambda \theta_t}_{\text{decoupled decay}}
\;-\; \alpha\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}.
$$
>
> 这能避免与自适应梯度统计的耦合带来的副作用，经验上更稳定。

In [7]:
class AdamWCustom(Optimizer):
    def __init__(
        self,
        params: Iterable[torch.nn.Parameter],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.01,
    ):
        if le < 0.0:
            raise ValueError(f"Invalid lr:{lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1:{betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2:{betas[1]}")
        if eps <= 0.0:
            raise ValueError(f"Invalid eps:{eps}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay:{weifht_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super.__init__(params, defaults)

### `__init__`：参数合法性与默认赋值

```python
def __init__(
    self,
    params: Iterable[torch.nn.Parameter],
    lr: float = 1e-3,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
):
    if lr < 0.0:
        raise ValueError(f"Invalid lr: {lr}")
    if not 0.0 <= betas[0] < 1.0:
        raise ValueError(f"Invalid beta1: {betas[0]}")
    if not 0.0 <= betas[1] < 1.0:
        raise ValueError(f"Invalid beta2: {betas[1]}")
    if eps <= 0.0:
        raise ValueError(f"Invalid eps: {eps}")
    if weight_decay < 0.0:
        raise ValueError(f"Invalid weight_decay: {weight_decay}")

    defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
    super().__init__(params, defaults)
```

* **类型与默认值**与 `torch.optim.AdamW` 对齐：

  * `lr` 学习率，>0；
  * `betas=(β1, β2)`：动量系数，常用 (0.9, 0.999)；
  * `eps`：数值稳定项，防止分母接近 0，默认1e-8；
  * `weight_decay`：解耦权重衰减系数 $\lambda$，默认0.01。
* **参数校验**：逐个抛 `ValueError`，保证训练时不会出现“隐性错误”。
* `defaults`：优化器基类会把它复制到每个 **param group**，从而可以支持**多组**参数、不同超参。

### `step`：执行一次参数更新

**AdamW 的一次 `step` 要做什么？**

**用当前梯度更新一阶/二阶动量 → 做偏置校正 → 用解耦权重衰减 & Adam 规则更新参数**。
真正计算时的步骤：

$$
\begin{aligned}
\text{(decoupled decay)}\quad
\theta_t &\leftarrow \theta_t - \alpha \lambda \theta_t \\
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat m_t &= \frac{m_t}{1-\beta_1^t},\qquad
\hat v_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t - \alpha \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
\end{aligned}
$$

其中 $g_t=\nabla_\theta \mathcal{L}(\theta_t)$，$\alpha$ 学习率，$\lambda$ 权重衰减系数，$\beta_1,\beta_2$ 动量指数衰减，$\epsilon$ 数值稳定项。

In [8]:
    @torch.no_grad()
    def step(self, closure: Optional[callable] = None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            weight_decay: floar = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("AdamWCustom does not support sparse gradients")

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state["exp_avg_sq"] = torch.zero_like(p, memory_format=torch.preserve_format)

                exp_avg: torch.Tensor = state["exp_avg"]
                exp_avg_sq: torch.Tensor = state["exp_avg_sq"]

                state["step"] += 1
                step: int = state["step"]

                if weight_decay != 0.0:
                    p.add_(p, alpha=-lr * weight_decay)

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1.0 - beta1 ** step
                bias_correction2 = 1.0 - beta2 ** step
                step_size = lr / bias_correction1

                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

                p.addcdiv_(exp_avg, denom, value=-step_size)
        return loss

**1. 装饰器与 closure**

```python
@torch.no_grad()
def step(self, closure: Optional[callable] = None):
    loss = None
    if closure is not None:
        with torch.enable_grad():
            loss = closure()
```

* `@torch.no_grad()`：**更新参数不需要被 autograd 记录**，否则会把优化步骤也纳入计算图，既慢又错。
* `closure`：pytorch中的一个函数，封装了 一次完整的前向 + 反向传播，是为了兼容像 LBFGS 这类“需要在 step 内重新计算 loss”的优化器接口保留的惯例。
  AdamW中用不到，这里只是为了保持和原版 pytorch 实现一致：若传入 `closure`，短暂 `enable_grad` 来重新前反传一次，拿到 `loss` 返回。


**2. 遍历参数组与超参获取**

```python
for group in self.param_groups:
    lr: float = group["lr"]
    beta1, beta2 = group["betas"]
    eps: float = group["eps"]
    weight_decay: float = group["weight_decay"]
```

* **param\_groups** 是 `torch.optim.Optimizer` 的核心机制，允许**不同层或张量用不同超参**（不同 lr / wd 等）。
* 读出本组的学习率、动量系数、eps、权重衰减。



**3. 遍历参数与梯度校验**

```python
for p in group["params"]:
    if p.grad is None:
        continue
    grad = p.grad

    if grad.is_sparse:
        raise RuntimeError("AdamWCustom does not support sparse gradients")
```

* `None` 冻结参数或本轮未参与计算的部分直接跳过。
* AdamW（官方实现也是）**不支持稀疏梯度**（稀疏需要特殊更新公式）。


**4. 状态（state）初始化 & 时间步**

```python
state = self.state[p]
if len(state) == 0:
    state["step"] = 0
    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
    state["exp_avg_sq"] = torch.zeros_like(p, memory_format=torch.preserve_format)

exp_avg: torch.Tensor = state["exp_avg"]
exp_avg_sq: torch.Tensor = state["exp_avg_sq"]

state["step"] += 1
step: int = state["step"]
```

* state 是一个**字典**，专门用来存储某个参数 p 在优化过程中的**历史状态信息**(动量、平方梯度、步数等)。
* 首次遇到参数：

  * `step=0`（马上会自增为 1）
  * `exp_avg`（$m_t$）初始 0 张量
  * `exp_avg_sq`（$v_t$）初始 0 张量
* `memory_format=torch.preserve_format`：**保留原参数内存布局**（例如 `channels_last`）。利于后续算子 kernel 最优化与缓存局部性。
* `step` 记录时间步 $t$，供**偏置校正**使用（$\beta^t$）。


**5. 解耦权重衰减（Decoupled Weight Decay）**

```python
if weight_decay != 0.0:
    p.add_(p, alpha=-lr * weight_decay)
```

* 这是 AdamW 和“把 L2 正则直接加到梯度里”的**关键区别**：

  * **解耦**：直接对参数做缩放 $\theta \leftarrow \theta - \alpha \lambda \theta = (1-\alpha\lambda)\theta$；
  * **耦合（L2 正则）**：把 $\lambda \theta$ 加到梯度上，交给后面的自适应分母去缩放，这会改变衰减在各维度上的相对强度（与 $v_t$ 相关），**经验上更不稳定**。
* 为什么放在动量更新**之前**？
  PyTorch AdamW 也是这么做的；从数值角度讲，放前或放后**几乎等价**（差在浮点细节）。

> 工程经验：很多训练(比如LayerNorm 和 bias 参数）会**关闭 LN/偏置项的衰减**（把它们放进一个 `weight_decay=0` 的组里），用 param\_groups 很容易做到。


**6. 指数滑动平均（EMA）的一阶/二阶动量**

```python
exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
```

* 公式对应：

  $$
  m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2.
  $$
* `addcmul_` 是 fused 原地：`exp_avg_sq += (1-β2) * (grad * grad)`；
  `mul_` / `add_` 都是**原地**，减少中间张量，省内存、快。
* **直觉**：

  * $m_t$：平滑的“平均梯度方向”
  * $v_t$：平滑的“各维度梯度方差”，用来**自适应**缩放学习率（维度越“剧烈”，步子越小）。


**7. 偏置校正（Bias Correction）**

```python
bias_correction1 = 1.0 - beta1 ** step
bias_correction2 = 1.0 - beta2 ** step
step_size = lr / bias_correction1
```

* 因为 $m_0=v_0=0$，前期 $m_t,v_t$ 有向 0 的偏置。校正项：

  $$
  \hat m_t = \frac{m_t}{1-\beta_1^t},\quad
  \hat v_t = \frac{v_t}{1-\beta_2^t}.
  $$
* 把 $1-\beta_1^t$ 放到 `step_size` 里（$\alpha/(1-\beta_1^t)$），把 $1-\beta_2^t$ 放到分母的 `sqrt` 里——**与官方实现完全等价**、更数值稳定。


**8. 分母（自适应因子）与最终更新**

```python
denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
p.addcdiv_(exp_avg, denom, value=-step_size)
```

* `denom = sqrt(v_hat) + eps`，其中 $v\_hat = v_t/(1-\beta_2^t)$。
  代码写成 $\sqrt{v_t}/\sqrt{1-\beta_2^t}$ 是一样的（更稳定的因式分解）。
* `add_(eps)`：$+\epsilon$ 防止分母接近 0（比如早期或梯度很小的情况）。
* `addcdiv_`：原地做 `p -= step_size * exp_avg / denom`。
  注意我们用的是**未除偏的一阶动量 `exp_avg`**，配合 `step_size = lr/(1-β1^t)`，这正是 $\alpha \hat m_t$。


**9. 返回 loss（如果有 closure）**

```python
return loss
```

* 若上面调用过 `closure`，把它的标量 `loss` 传回去，方便上层记录。

In [9]:
def get_adamw_cls():
    return AdamWCustom

返回“我们自己实现的AdamW 类”，语义上是对其 torch.optim.AdamW 的。

## 4.2 lr_cosine_schedule

虽然这里函数名只有 cosine_schedule ，并且 pytorch 官方的余弦学习率中也没有，但是实现要求里说明了需要带 linear warmup。
```python
"""
Given the parameters of a cosine learning rate decay schedule (with linear
warmup) and an iteration number, return the learning rate at the given
iteration under the specified schedule.
"""
```

**1. 什么是 warmup?**

**warmup** 指 **学习率预热**（learning rate warmup）：在训练的最初几个epoch，学习率不会直接设到目标的最大值，而是**从一个较小的值开始，逐渐增大到预设的最大学习率**。

常见形式：

* **线性 warmup**：学习率从 0 线性增加到最大值。
* **指数 warmup**：学习率按指数规律增加。
* **常见实践**：先 warmup 几百或几千个 step，再进入正常的学习率调度（例如余弦衰减、step decay 等）。



**2. 为什么要用 warmup？**

训练初期模型参数是随机初始化的，网络输出和梯度分布都很不稳定，如果一开始就用一个**很大的学习率**，容易造成：

* **梯度爆炸 / 发散**：初始梯度方向杂乱，大学习率会导致参数更新过大，训练不稳定；
* **损失震荡甚至 nan**：尤其是 Transformer / 大模型，层归一化 + 残差结构容易放大这种不稳定性；
* **优化器的动量未收敛**：像 Adam/AdamW 等自适应优化器一开始的动量估计还不准，大步更新会导致方向错误。

因此 warmup 的核心目的：

* **稳定训练初期的优化过程**
* **让优化器的动量有时间收敛**
* **避免大学习率直接冲击随机初始化参数**



**3. 举个例子**

假设我们设定最大学习率 = 0.001

* 如果没有 warmup，第一步就直接 lr=0.001，梯度可能把参数“甩飞”，训练不稳定。
* 如果用 1000 step 的线性 warmup：

  * step=1 → lr=0.000001
  * step=500 → lr=0.0005
  * step=1000 → lr=0.001
  * 之后才开始余弦衰减。

这样，前 1000 step 相当于“热身”，让模型逐渐适应大学习率。

In [10]:
def get_lr_cosine_schedule(
    *,
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
) -> float:
    if it <= warmup_iters:
        if warmup_iters <= 0:
            return float(max_learning_rate)
        return float(max_learning_rate) * (it / float(warmup_iters))

    span = max(1, cosine_cycle_iters - warmup_iters)
    progress = (it - warmup_iters) / float(span)
    if progress >= 1.0:
        return float(min_learning_rate)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    lr = min_learning_rate + (max_learning_rate - min_learning_rate) * cosine

    if lr < min_learning_rate:
        lr = min_learning_rate
    if lr > max_learning_rate:
        lr = max_learning_rate

    return float(lr)

**1. 线性 warmup（含端点）**

```python
    # 线性 warmup（含端点）
    if it <= warmup_iters:
        if warmup_iters <= 0:
            return float(max_learning_rate)
        return float(max_learning_rate) * (it / float(warmup_iters))
```

* 当 `it` 还没超过 warmup 终点：

  * 特判 `warmup_iters <= 0`：这表示**根本不做 warmup**。这里直接返回 `max_learning_rate`，等价于“warmup 长度为 0 时从第一步就用 `max_lr`”（避免除零）。
  * 正常情况：线性插值

    $$
    \text{lr}(it) = \alpha_{\max} \cdot \frac{it}{T_w},\quad 0 \le it \le T_w.
    $$

    * `it=0` ⇒ `0`；`it=warmup_iters` ⇒ `max_learning_rate`。
    * “包含端点”的处理让 **warmup 结束那一刻**恰好到达峰值 LR，和社区常见实现一致。


**2. 余弦退火**

```python
    # 余弦段
    span = max(1, cosine_cycle_iters - warmup_iters)  # 防止除零
    progress = (it - warmup_iters) / float(span)
    if progress >= 1.0:
        return float(min_learning_rate)
```

* 进入余弦段的前提是 `it > warmup_iters`（上一段已 return 结束），此时需要把 `it` 归一化到 $(0,1)$。
* `span = max(1, T_c - T_w)`：

  * 余弦段长度是 `T_c - T_w`。若写错参数导致 `T_c <= T_w`，这里用 `max(1, …)` 防止除零——**稳健性**考虑。
* `progress = (it - T_w) / span`：

  * 当 `it = T_w + 1` 时 `progress` 接近 `0^+`；
  * 当 `it` 逐步逼近 `T_c`，`progress → 1^-`。
* `if progress >= 1.0:`：

  * 当 `it >= T_c`（或某些极端舍入）直接落到**尾段**：返回 `min_learning_rate`。

```python
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    lr = min_learning_rate + (max_learning_rate - min_learning_rate) * cosine
```

* 经典的**半周期余弦**系数（从 1 平滑到 0）：

  $$
  \text{cosine}(t) = \tfrac{1}{2}\left(1 + \cos(\pi t)\right),\quad t\in[0,1].
  $$

  * `t=0` ⇒ `cosine=1`，学习率等于 `max_lr`；
  * `t=1` ⇒ `cosine=0`，学习率等于 `min_lr`。
* 线性映射到区间 $[\alpha_{\min}, \alpha_{\max}]$：

  $$
  \text{lr}(it) = \alpha_{\min} + (\alpha_{\max}-\alpha_{\min}) \cdot \text{cosine}(\text{progress}).
  $$
* 这保证了**连续性**：在 warmup 末端到余弦起点，LR 值连续（不过导数在分段点一般**不连续**，这是常见且可接受的设计）。


**3. 数值钳制与返回**

```python
    # 数值钳制，避免浮点误差越界
    if lr < min_learning_rate:
        lr = min_learning_rate
    if lr > max_learning_rate:
        lr = max_learning_rate
    return float(lr)
```

* 由于浮点误差，`cos`/除法可能让结果在端点附近出现极细小的**越界**（例如应为 0.1 却算成 0.09999999997）。
* 这里对 LR 做**上下界钳制**，保证严格落在 $[\alpha_{\min}, \alpha_{\max}]$。
* `float(lr)`：确保返回的是 Python `float`（不是 numpy 标量或别的类型）。这在某些日志/序列化路径上更稳。

## 4.3 测试

使用 
```bash
uv run -- pytest -x -vv -k optimizer
``` 
进行测试即可

# 5 data

这一部分是实现：    
get_batch

就是实现：从一个长的一维 dataset 流中，随机抽取 batch_size 个连续窗口，每个窗口长度为 context_length；X 是窗口本身，Y 是右移一位（next-token 预测的标签）     
关于 pin_memory：
1. 关闭 pin_memory：在 GPU 显存足够的情况下，可以关闭 pin_memory，该方法将 dataset 一次性全部加载到 GPU 上所有的计算都是在 GPU 上完成，CPU-GPU 的数据传输仅进行一次，所以性能最好；
2. 开启 pin_memory：在 GPU 显存不足以容纳整个数据集的情况下，将数据集保留在 CPU 主内存中，每次只在 CPU 上准备好一个小批量的数据，然后仅将这个小批量数据传输到 GPU。该方法引入了每一步的 CPU-GPU 数据传输开销，但大大降低了对显存的需求，保证了程序的可用性。此外，通过使用 pin_memory（锁页内存）的优化方法，可以使数据传输异步进行，从而与 GPU 上的计算并行，能最大限度地隐藏数据传输带来的延迟。

In [11]:
def get_batch(
    *, dataset_np, batch_size: int, context_length: int,
    device: str, pin_memory: bool = False
):
    is_cuda = str(device).startswith("cuda")
    if pin_memory and not is_cuda:
        raise ValueError("pin_memory=True only makes sense when device is CUDA")

    if pin_memory:
        # CPU + pinned 路径
        toks = torch.as_tensor(dataset_np, dtype=torch.long, device="cpu").pin_memory()
        idx_device = "cpu"
    else:
        # 直接放目标设备
        toks = torch.as_tensor(dataset_np, dtype=torch.long, device=device)
        idx_device = device  # 索引必须与 toks 在同设备

    n = toks.numel()
    if n < context_length + 1:
        raise ValueError(f"need at least {context_length+1}, got {n}")

    max_start = n - context_length - 1
    starts = torch.randint(0, max_start + 1, (batch_size,), device=idx_device)
    ar = torch.arange(context_length, device=idx_device)
    idx = starts[:, None] + ar[None, :]
    X = toks[idx]
    Y = toks[idx + 1]

    if pin_memory:
        # 仅在 pinned 路径下做异步 H2D
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

    return X, Y

## 5.2 测试

使用 
```bash
uv run -- pytest -x -vv -k data
``` 
进行测试即可

# 6 I/O Checkpoints

这一部分是实现：    
1. save_checkpoint
2. load_checkpoint

## 6.1 _is_pathlike

区分“路径”和“文件对象”

In [12]:
def _is_pathlike(x) -> bool:
    return isinstance(x, (str, bytes, os.PathLike))

## 6.2 save_checkpoint

**把训练的中间状态保存到 checkpoint 文件**，方便后续恢复。

保存的内容包括：

1. **模型参数 (`model.state_dict()`)**
   只包含模型里权重和 buffer（例如 `LayerNorm.running_mean`），而不是整个模型对象。这样更稳健，跨版本/跨路径都能加载。

2. **优化器状态 (`optimizer.state_dict()`)**
   包含动量项、平方梯度累积等（Adam/AdamW 中的 `exp_avg`、`exp_avg_sq`），否则恢复训练时梯度动态会断裂。

3. **当前迭代数 (`iteration`)**
   用来恢复训练循环的步数、学习率调度器位置、日志编号等。

              
> 这里说明一下为什么直接用 torch.save()，作业说明 cs336_spring2025_assignment1_basics.pdf 中明确说了对于模型原理相关的 torch.nn, torch.nn.functional, 或 torch.optim 中的大部分定义不能使用，但是5.2部分说了torch.save(obj, dest) can dump an object ... which can then be loaded back into memory with torch.load(src). 所以是没问题的。

In [13]:
def save_checkpiont(
    *, 
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    iteration: int,
    out: str | os.PathLike | BinaryIO | IO[bytes],
) -> None:
    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "iteration": int(iteration),
    }
    if _is_pathlike(out):
        with open(out, "wb") as f:
            torch.save(payload, f)
    else:
        torch.save(payload, out)

## 6.3 load_checkpoint

1. **读取 checkpoint 文件**

   * 支持文件路径（字符串、`Path`）
   * 也支持已经打开的二进制文件对象（例如 `open(..., "rb")` 或 `io.BytesIO()`）

   通过 `torch.load(..., map_location="cpu")` 把保存的内容读进来（先放到 CPU，保证跨设备加载安全）。

2. **恢复训练状态**

   * 用 `model.load_state_dict(payload["model_state"])` 把模型的参数恢复到保存时的值。
   * 用 `optimizer.load_state_dict(payload["optimizer_state"])` 把优化器的动量、学习率组等内部状态也恢复回来。
     这样模型和优化器都会回到**和保存时一模一样的状态**。

3. **返回迭代计数**

   * 从 checkpoint 中取出 `iteration` 并返回。
   * 训练循环里就能接着跑，比如从第 1000 步恢复继续往下训练。

In [14]:
def load_checkpoint(
    *,
    src: str | os.PathLike | BinaryIO | IO[bytes],
    model: torch.nn.Model,
    optimizer: torch.optim.Optimizer,
) -> int:
    if _is_pathlike(src):
        with open(src, "rb") as f:
            payload = torch.load(f, map_location = "cpu")
    else:
        payload = torch.load(src, map_location = "cpu")

    model.load_state_dict(payload["model_state"])
    optimizer.load_state_dict(payload["optimizer_state"])

    return int(payload["interation"])

## 6.4 测试

使用 
```bash
uv run -- pytest -x -vv -k serialization
``` 
进行测试即可

# 7 Tokenizer



这一部分实现的是：

**BPE（Byte Pair Encoding）**：**字节对编码**算法。我们要创建一个“分词器”（Tokenizer）。语言模型（比如GPT）不直接理解文字，它们只理解数字。分词器的任务就是把人类的语言（比如字符串 "Hello, world!"）转换成一串数字（比如 [15496, 11, 995, 0]），并且能再把这串数字转换回原来的文字。     

BPE 的**核心思想**：一开始把所有单个的字符当作基础词汇，然后不断地寻找最常出现的相邻“词对”，把它们合并成一个新的“词”。重复这个过程，就能学到像 "ing", "tion" 这样常见的词根或单词片段。这样既能有效压缩词汇表大小，又能处理没见过的词（OOV, Out-of-Vocabulary a problem）。

如果想要深入了解 BPE，非常推荐去看这个教程：[Let's build GPT: from scratch, in code, spelled out.](https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLwHkaSK8WYmI67jO5EEYtzTyVOT9i5GWB&index=2)

In [15]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

这是 **GPT-2 分词器的预分词（pre-tokenization）正则表达式**。它的作用是：把原始文本切成“初步的词块”（pieces），然后再交给 BPE 规则去进一步合并。

1. **`'(?:[sdmt]|ll|ve|re)`**

   * 匹配英语里的常见缩写后缀，例如：

     * `I'm` → `'m`
     * `you're` → `'re`
     * `they've` → `'ve`
     * `she'll` → `'ll`
     * `it's` → `'s`
   * 用 `?:` 表示非捕获分组。

2. **` ?\p{L}+`**

   * `\p{L}` 表示 Unicode 中的“字母”（Letter）。
   * `+` 表示连续的一个或多个字母。
   * 前面的 ` ?` 允许前面有一个空格。
   * 意义：**字母串（单词），并保留开头空格**。
   * 例子：

     * `"Hello"` → `"Hello"`
     * `" world"` → `" world"`

3. **` ?\p{N}+`**

   * `\p{N}` 表示 Unicode 中的“数字”（Number）。
   * 类似上面的逻辑，表示一串数字，可以前面带一个空格。
   * 例子：

     * `"123"` → `"123"`
     * `" 42"` → `" 42"`

4. **` ?[^\s\p{L}\p{N}]+`**

   * 匹配“非空格、非字母、非数字”的连续字符（符号/标点），前面允许有一个空格。
   * 例子：

     * `","` → `","`
     * `" !!!"` → `" !!!"`

5. **`\s+(?!\S)`**

   * `\s+` 表示一个或多个空白字符。
   * `(?!\S)` 表示**后面不是非空白**，即这些空格后面没有别的东西了（字符串结尾）。
   * 意义：匹配**尾部的空格**。

6. **`\s+`**

   * 匹配一般的空格（如果前面几条规则都没处理）。

**总结**

这个正则的作用是把文本切分成几类 token 初始块：

1. **缩写后缀**：`'s`, `'ll`, `'ve`, `'re`, `'m`, `'t`, `'d`
2. **单词**：前导空格 + 字母串
3. **数字**：前导空格 + 数字串
4. **符号/标点**：前导空格 + 一串标点/符号
5. **尾部空格**
6. **其他空格**

**举个例子**

输入文本：

```
Hello, I'm 25 years old!  
```

经过这个正则分词：

```
["Hello", ",", " I", "'m", " 25", " years", " old", "!", "  "]
```

之后这些块会被转换成字节序列，再交给 BPE 合并。

## 7.1 __init__

```python
# 1. 存储基础数据
self.vocab = vocab
self.encoder = {b: i for i, b in vocab.items()}
```
1.  **`self.vocab` 和 `self.encoder`**:

      * `vocab`: “解码器”，一个从数字ID到字节（bytes）的字典。当我们有了一个ID，比如 `50256`，可以用 `vocab[50256]` 查到它代表的字节。
      * `encoder`: “编码器”，它正好和`vocab`相反。我们用它来把字节（比如 `b'<|endoftext|>'`）转换成对应的ID。这里用了一个字典推导式 `{b: i for i, b in vocab.items()}` 来快速创建这个反向映射。这样，编码和解码的查询都非常快。

```python
self.bpe_ranks = {pair: i for i, pair in enumerate(merges)}
```
2.  **`self.bpe_ranks`**:

      * `merges` 是一个列表，包含了所有合并规则，并且是按学习顺序排列的。例如 `[(b'e', b'n'), (b'en', b'd')]`。
      * 因为在编码时需要频繁地查找需要优先合并的词对。在列表中搜索效率很低,可以通过将其转换成一个字典 `bpe_ranks`，键是词对 `(b'e', b'n')`，值是它在列表中的索引 rank。**rank 越小，优先级越高**。这样可以将查找一个词对的优先级的时间复杂度变成 O(1) 。

```python
self.special_tokens = special_tokens or []
self.special_tokens_encoder: dict[str, int] = {}
self.special_tokens_decoder: dict[int, str] = {}
``` 
3.  **`self.special_tokens` 和 `self.special_tokens_encoder/decoder`**:

      * 有一些特殊 token 我们不希望被BPE算法拆分，比如 `"<|endoftext|>"` (文本结束符)。
      * 将这些特殊 token 单独存起来，并创建一个专用的编码器和解码器。

```python
self.pat = re.compile(PAT)
self.special_pat = None
```
4.  **`self.pat = re.compile(PAT)`**:

      * `PAT` 前面已经说了，这里不再赘述。
      * `re.compile()` 会预编译这个正则表达式。（如果我们要在一个循环里反复使用同一个正则表达式，预编译可以大大提高速度）

```python
if self.special_tokens:
    self._setup_special_tokens()
```
5.  **`self._setup_special_tokens()`**:

      * 专门用来处理特殊 token,我们稍后会详细看它。

In [16]:
class Tokenizer:
    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[tuple[bytes, bytes]],
        spcial_tokens: list[str] | None = None,
    ):
        self.vocab = vocab
        self.encoder = {b: i for i, b in vocab.items()}
        self.bpe_ranks = {pair: i for i, pair in enumerate(merges)}
        self.special_tokens = special_tokens or []
        self.special_tokens_encoder: dict[str, int] = {}
        self.special_tokens_decoder: dict[int, str] = {}
        self.pat = re.compile(PAT)
        self.special_pat = None
        if self.special_tokens:
            self._setup_special_tokens()

## 7.2 _setup_special_tokens

```python
sorted_special_tokens = sorted(self.special_tokens, key=len, reverse=True)
```
1. **`sorted_special_tokens`**:

   * 将所有的特殊 token 按照长度进行排序，保证子集重叠的 token 能够先处理较长一条，比如：

      * `"<|eot|>"`
      * `"<|eot|><|eot|>"`

   * 那么必须先匹配 **更长的** `" <|eot|><|eot|>"`，否则会被前者提前切开。
   * 所以正则里优先级由长度保证。

```python
# 将特殊token添加到词汇表中
for token_str in sorted_special_tokens:
    token_bytes = token_str.encode("utf-8")
    if token_bytes not in self.encoder:
        # 分配一个新的ID
        new_id = len(self.vocab)
        self.vocab[new_id] = token_bytes
        self.encoder[token_bytes] = new_id
    
    # 存储特殊token的字符串到ID的映射
    self.special_tokens_encoder[token_str] = self.encoder[token_bytes]
    self.special_tokens_decoder[self.encoder[token_bytes]] = token_str
```
2. **把特殊 token 加入词表，并存储其映射**

   * 首先将新的 token 编码
   * 然后为其分配一个新的 id（因为词表 `self.vocab` 原本是 **id → bytes**，`self.encoder` 是 **bytes → id**），新 id = 当前 vocab 的长度
   * 最后再将特殊字符串 **id → str** 和 **str → id** 的映射保存到 special_tokens_encoder 和 special_tokens_decoder

> 这里解释一下，特殊字符之所以特殊是因为没办法直接用 utf-8 进行编解码，所以需要存储一个专门的编码解码器用来对其进行编码和解码。

```python
# 创建一个正则表达式，用于根据特殊token分割输入文本
# 使用re.escape来处理可能包含正则表达式元字符的特殊token
special_pattern = "|".join(re.escape(st) for st in sorted_special_tokens)
self.special_pat = re.compile(f"({special_pattern})")
```
3. **构造正则，用来切分文本**

   * 构建一个正则表达式，这个表达式能一次性在文本中找到所有我们定义的特殊 token 。
   * 当使用一个正则表达式去调用 re.split() 时，它不仅会按匹配到的内容分割字符串，还会把匹配到的内容（也就是我们的特殊token）本身也保留在结果列表里

In [17]:
    def _setup_special_tokens(self) -> None:
        sorted_special_tokens = sorted(self.special_tokens, key=len, reverse=True)
        for token_str in sorted_special_token:
            token_bytes = token_str.encode("utf-8")
            if token_bytes not in self.encoder:
                new_id = len(self.vocab)
                self.vocab[new_id] = token_bytes
                self.encoder[token_bytes] = new_id
            self.special_tokens_encoder[token_str] = self.encoder[token_bytes]
            self.special_tokens_decoder[self.encoder[token_bytes]] = token_str
        special_pattern = "|".join(re.escape(st) for st in sorted_special_tokens)
        self.special_pat = re.compile(f"({special_pattern})")

## 7.3 _get_pairs

把输入的 token 列表中相邻的两个元素组成 pair，并返回所有 pair 的集合。

In [18]:
    @staticmethod
    def _get_pairs(tokens: list[bytes]) -> set[tuple[bytes, bytes]]:
        return set(zip(tokens, tokens[1:]))

## 7.4 _bpe_merge

1. **`pairs = self._get_pairs(tokens)`**

   找到所有相邻的对

2. **`best_pair = min(pairs, key=lambda p: self.bpe_ranks.get(p, float("inf")))`**

   找到优先级最高的对

3. **`if best_pair not in self.bpe_ranks:`**

   如果没有可以合并的对了，就结束

4. **执行合并**

```python
new_tokens = []
i = 0
while i < len(tokens):
    if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
        new_tokens.append(tokens[i] + tokens[i+1])
        i += 2
    else:
        new_tokens.append(tokens[i])
        i += 1
tokens = new_tokens
```

5. **`return [self.encoder[token] for token in tokens]`**

   将最终的字节列表转换为ID列表

In [19]:
    def _bpe_merge(self, piece: bytes) -> list[int]:
        tokens = [bytes([b]) for b in piece]
        while True:
            pairs = self._get_pairs(tokens)
            if not pairs:
                break
            best_pair = min(pairs, key=lambda p: self.bpe_ranks.get(p, float("inf")))
            if best_pair not in self.bpe_ranks:
                break
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
                    new_tokens.append(tokens[i] + tokens[i+1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return [self.encoder[token] for token in tokens]

## 7.5 encode

1. **根据特殊token分割文本**

```python
if self.special_pat is None:
    chunks = [text]
else:
    chunks = self.special_pat.split(text)
```

根据特殊 token 分割文本

2. **合并**

```python
for chunk in chunks:
    # ... 空块处理 ...
    
    # 2. 判断是特殊token还是普通文本
    if chunk in self.special_tokens_encoder:
        token_ids.append(self.special_tokens_encoder[chunk])
    else:
        # 3. 对普通文本进行预分词
        pre_tokens = self.pat.findall(chunk)
        for pre_token in pre_tokens:
            # 4. 对每个小块进行BPE合并
            token_ids.extend(self._bpe_merge(pre_token.encode("utf-8")))
```

In [20]:
    def encode(self, text: str) -> list[int]:
        if not text:
            return []
        token_ids = []
        if self.special_pat is None:
            chunks = [text]
        else:
            chunks = self.special_pat.split(text)
        for chunk in chunks:
            if not chunk:
                continue
            if chunk in self.special_tokens_encoder:
                token_ids.append(self.special_tokens_encoder[chunk])
            else:
                pre_tokens = self.pat.findall(chunk)
                for pre_token in pre_tokens:
                    token_ids.extend(self._bpe_merge(pre_token.encode("utf-8")))
        return token_ids

## 7.6 encode_iterable

```python
buffer = ""
for chunk in iterable:
    buffer += chunk
    for token_id in self.encode(buffer):
        yield token_id
    buffer = ""
```

当我们处理一个非常大的文本文件时不能直接用 encode 读取整个文件（会内存溢出）

```python
if buffer:
    for token_id in self.encode(buffer):
        yield token_id
```
简单的保护机制，一般不会触发

In [21]:
    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        buffer = ""
        for chunk in iterable:
            buffer += chunk
            for token_id in self.encode(buffer):
                yield token_id
            buffer = ""
        if buffer:
            for token_id in self.encode(buffer):
                yield token_id

## 7.7 decode

用解码器 self.vocab 将每个 id 转换回 bytes，然后再将 bytes 解码为 str

In [22]:
    def decode(self, ids: list[int]) -> str:
        if not ids:
            return ""
        token_bytes_list = [self.vocab.get(i, b"") for i in ids]
        full_bytes = b"".join(token_bytes_list)
        return full_bytes.decode("utf-8", errors="replace")

## 7.8 from_files

就是将手动传入 vocab=vocab, merges=merges, special_tokens=special_tokens 这几个参数变成了从文件中读入

In [23]:
    @classmethod
    def from_files(
        cls,
        vocab_filepath: str | os.PathLike,
        merges_filepath: str | os.PathLike,
        special_tokens: list[str] | None = None,
    ) -> "Tokenizer":
        with open(vocab_filepath, "r", encoding="utf-8") as f:
            vocab_json = json.load(f)
            vocab = {int(k): v.encode("utf-8") for k, v in vocab_json.items()}
        merges = []
        with open(merges_filepath, "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("#") or not line.strip():
                    continue
                p1, p2 = line.strip().split()
                merges.append((p1.encode("utf-8"), p2.encode("utf-8")))
        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

## 7.9 测试

使用 
```bash
uv run -- pytest -x -vv -k tokenizer
``` 
进行测试即可

# 8 train_bpe

**`train_bpe.py` 和 `tokenizer.py` 是什么关系？**        
答：两者是**生产者与消费者**的关系，它们是构建和使用BPE分词器过程中前后衔接、相辅相成的两个独立部分。简单来说：

  * **`train_bpe.py` 是“训练器”或“构建者”**。它的任务是一次性地分析一个大型文本语料库，学习出一套最高效的文本压缩规则。
  * **`tokenizer.py` 是“使用者”或“执行者”**。它加载“训练器”产出的规则，并应用这些规则来对任何新的文本进行快速的编码和解码。

## 8.2 测试

使用 
```bash
uv run -- pytest -x -vv -k train_bpe
``` 
进行测试即可

由于本人不是 NLP 相关研究方向的，所以这一部分感兴趣的可以自己研究一下，这里就不做过多讲解了。

# 9 model

这一部分需要实现6个模块：

- attention
- embedding
- feed_worward
- linear
- normalization
- transformer

这一部分的设计思路就是将 Transformer 按照不同的模块进行拆分，然后分别实现即可。

## 9.1 Linear

线性变换，类似于 torch.nn.Linear，但是没有偏置项 bias，因为[作业说明](../../cs336_spring2025_assignment1_basics.pdf)中明确说明了省略偏置项。

> 因为在深层的 transformer 结构中 bias 基本都是冗余的，现在大模型基本都移除了 bias

### 9.1.1 __init__

`in_features`、`out_features`: 输入输出的维度    
`factory_kwargs`：张量存储的设备和类型，用于后面初始化权重参数使用    
`torch.empty((out_features, in_features), **factory_kwargs)`:创建一个形状为(out_features, in_features)的张量，存储设备和张量存储类型都存在`factory_kwargs`中，然后通过`nn.Parameter`将其包裹起来，这样在训练的时候 Pytorch 便会计算它的梯度，并用优化器来更新它。    
`self.reset_parameters()`:在定义好了权重参数之后，调用 reset_parameters 方法，用一种特定的策略来填充我们刚刚创建的那个“空”张量，赋予它有意义的初始值，以确保训练从合适的初始值开始。

In [ ]:
class Linear(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        factory_kwargs = {"device": device, "dtype": dtype}
        self.weight = nn.Parameter(torch.empty((out_features, in_features), **factory_kwargs))
        self.reset_parameters()

### 9.1.2 reset_parameters

给模型的权重（self.weight）一个良好、合理的初始值。   

**为什么不直接让它全是 0 或者全是随机值？**    
**答**：因为一个好的初始值对神经网络的训练至关重要。如果初始值太差，可能会导致训练过程非常缓慢，甚至完全失败（梯度消失或梯度爆炸）。

这里使用的是 **Glorot Normal (Xavier Normal) 初始化** 的 **截断正态分布** 版本。

> 关于初始化常用的有很多，这里我只是根据作业要求来进行实现的，这方面没有专门进行了解，就不做过多说明了

In [25]:
    def reset_parameters(self) -> None:
        std = math.sqrt(2 / (self.in_features + self.out_features))
        nn.init.trunc_normal_(self.weight, mean=0.0, std=std, a=-3*std, b=3*std)

### 9.1.3 forward

这一步是进行前向传播。    
**`x`** 的形状为：[batch_size, seq_length, d_in]    
**`weight`** 的形状为：[d_out, d_in]    

**`einsum`** 为 **爱因斯坦求和约定（Einstein summation convention）** ，它的核心思想是：
- 只写出 **索引关系**（谁和谁相乘、在哪些维度上求和、保留哪些维度），框架自动帮你做张量乘法与求和。

**`einops`**（Einstein Operations for Tensors）库的 `einsum` 是对这个概念的统一接口，能无缝在 **NumPy、PyTorch、TensorFlow、JAX、MXNet** 等框架里使用。

特别是注意力机制的计算中非常方便，看下面例子：

```python
Q = torch.randn(32, 8, 50, 64)  # batch, heads, seq_len, dim
K = torch.randn(32, 8, 50, 64)

# 计算注意力分数 (Q * K^T)
scores = einsum("b h i d, b h j d -> b h i j", Q, K)
```

所以
```python
ensum(x, self.weight, "... d_in, d_out d_in -> ... d_out")
```
就是用 `x` 中的每一个 batch_size 去和 weight 做矩阵乘，计算输出.

In [26]:
    def forward(self, x: Float[Tensor, "... d_in"]) -> Float[Tensor, "... d_out"]:
        return einsum(x, self.weight, "... d_in, d_out d_in -> ... d_out")

### 9.1.4 extra_repr

当我们使用 `print(model)`、`repr(model)`、或把定义的模型放进 `nn.Sequential` 打印网络结构时，PyTorch 会先打印模块名（`Linear`），再把 `extra_repr()` 返回的字符串放进括号里打印出来

作用：

* 默认的 `Module.__repr__` 只会打印层级结构；如果你不重写 `extra_repr()`，通常会看到 `Linear()`（不含任何超参数信息）。
* 重写后，你能直观看到该层的**关键超参数**，便于调试与排错。

In [27]:
    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias=False"

# 9.2 Embedding



`Embedding` 模块的作用是将输入的整数索引（token IDs）转换为密集向量（dense vectors）。相当于是一个大型的查找表（lookup table），这个表是一个矩阵，每一行代表一个 token 的向量表示。当模型接收到一个 token ID（比如 42），它就会去查找这个表的第 42 行，并返回该行对应的向量。

### 9.2.1 __init__

`num_embeddings`:嵌入的数量，也就是词汇表的大小，决定了 `self.weight` 的行数。
`embedding_dim`:嵌入维度，也就是每一个 token ID 被编码成的向量的长度，决定了 `self.weight` 的列数。

In [ ]:
class Embedding(nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        embedding_dim:int,
        device:torch.device | str | None = None,
        dtype:torch.dtype | None = None
    ):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        factory_kwargs = {"device": device, "dtype": dtype}
        self.weight = nn.Parameter(torch.empty((num_embeddings, embedding_dim), **factory_kwargs))
        self.reset_parameters()

### 9.2.2 reset_parameters

按照作业要求进行模型参数初始化。

In [29]:
    def reset_parameters(self) -> None:
        std = 1.0
        nn.init.trunc_normal_(self.weight, mean=0.0, std=std, a=-3*std, b=3*std)

### 9.2.3 forward

返回 token_ids 所对应的编码向量。

In [30]:
    def forward(self, token_ids: Int[Tensor, "..."]) -> Float[Tensor, "... d_model"]:
        return self.weight[token_ids]

## 9.3 RoPE

**1. 问题：Transformer 为何需要位置信息？**

标准的 Transformer 模型中的自注意力机制（Self-Attention）是**位置无关的**。也就是说，它处理句子 "猫坐在垫子上" 和 "垫子坐在猫上" 的方式是一样的，因为它只是将所有词元（token）视为一个无序的集合。这显然是不行的，因为词序在语言中至关重要。

**2. 传统解决方案：绝对位置编码**

最初的 Transformer 通过给每个 token 的词向量**加上**一个代表其绝对位置（第0个、第1个、第2个...）的"位置向量"来解决这个问题。

  * **优点**：简单直接。
  * **缺点**：泛化能力差。如果模型训练时最大长度是512，那么它完全不知道第513个位置应该是什么样的，导致处理更长序列时性能下降。

**3. RoPE 的核心思想：用“相对位置”代替“绝对位置”**

**关键点**：自注意力的核心是计算查询向量 $q\_m$（在 m 位置）和键向量 $k\_n$（在 n 位置）的点积，这个点积决定了它们之间的注意力分数。所以可以设计一种编码方式，让 $q\_m$ 和 $k\_n$ 的点积结果**只与它们的相对位置 $(m-n)$ 有关**即可。

RoPE 的天才之处在于，它发现可以通过**复数**或者等价的**向量旋转**来实现这个目标。

  * **核心操作**：RoPE **不往**词向量上加任何东西，而是根据 token 的位置，对其进行**旋转**。
  * **如何旋转**：
    1.  将 $d\_k$ 维的向量（比如 $q$ 或 $k$）两两一组，看作是 $\\frac{d\_k}{2}$ 个二维向量（或者说，复数）。
    2.  对于一个在位置 $m$ 的 token，RoPE 会用一个只与 $m$ 相关的旋转角度 $\\theta\_m$ 来旋转这些二维向量。
    3.  同理，对于在位置 $n$ 的 token，就用一个只与 $n$ 相关的旋转角度 $\\theta\_n$ 来旋转。

这里推荐几个讲解的帖子，讲的都非常好：

* [RoPE旋转位置编码原理解读](https://www.bilibili.com/video/BV1AN4y1X7SK/?share_source=copy_web&vd_source=608471d0e25c02d240b92470bd78f213)
* [Transformer升级之路：2、博采众长的旋转式位置编码](https://kexue.fm/archives/8265)
* [一步一步，推导旋转位置编码 (Rotary Position Embedding, RoPE)](https://zhuanlan.zhihu.com/p/644585013)

### 9.3.1 __init__

```python
d_k % 2
```       
* RoPE 将所有元素按 2 个一组组成复数，所以 d_k 必须是偶数。   

```python
freqs = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=device).float() / d_k))
```

* 构造每个二维平面的**角频率** $\omega_i$。
  `torch.arange(0, d_k, 2)` 产生 0,2,4,...（一共 d\_k/2 个），再除以 d\_k 得到 $\frac{2i}{d_k}$。
  $ \omega_i = \theta^{-2i/d_k} = 1 / \theta^{(2i/d_k)}$。
  `freqs` 形状是 `[d_k/2]`。


```python
t = torch.arange(max_seq_len, device=device)
freqs = torch.outer(t, freqs)
```

* `t` 是位置索引 0..max\_seq\_len-1。
* 外积得到角度矩阵 $\Theta \in \mathbb{R}^{\text{max\_seq\_len} \times (d_k/2)}$，其中 $\Theta[t, i] = t \cdot \omega_i$。

```python
freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
```

* 把角度矩阵转成复平面上的单位向量 $e^{j\Theta[t,i]} = \cos\Theta + j\sin\Theta$。
* `torch.polar(r, theta)` 以极坐标构造复数，半径 `r=1`，角度为 `freqs`。
* `freqs_cis` 形状 `[max_seq_len, d_k/2]`，dtype 为 **complex64**（或 complex32/complex128 视 PyTorch 与输入 dtype）。

```python
self.register_buffer("freqs_cis", freqs_cis, persistent=False)
```

* 把预计算的旋转因子注册为 **buffer**（随模型搬设备、无需参与梯度）。
* `persistent=False` 表示 **不写入 state\_dict**，减少 checkpoint 体积；加载后可按需要重建。

In [31]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(
        self,
        d_k: int,
        max_seq_len: int,
        theta: float = 10000.0,
        device: torch.device | str | None = None,
    ):
        super().__init__()
        if d_k % 2 != 0:
            raise ValueError("d_k must be even for Rotary Positional Embedding.")
        freqs = 1.0 / (theta ** (torch.arange(0, d_k, 2, device=device).float() / d_k))
        t = torch.arange(max_seq_len, device=device)
        freqs = torch.outer(t, freqs)
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
        self.register_buffer("freqs_cis", freqs_cis, persistent=False)

### 9.3.2 forawrd

```python
def forward(self, x: Float[Tensor, "... seq_len d_k"], token_positions: Int[Tensor, "... seq_len"]):
```

* `x`：通常是 Q 或 K，形状可以是 `(B, H, T, d_k)` 或更一般的 `...`。
* `token_positions`：同样是 `... seq_len` 的整型索引，指明每个位置在全局时间轴上的“绝对位置”（便于增量解码时继续往后接）。

```python
x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
```

* 把最后一维 `d_k` 变成两份：`(d_k/2, 2)`，再用 **view\_as\_complex** 把每对 `(real, imag)` 视作一个复数。
* 这一步要用 `float()`（float32）是因为半精度/混合精度下，复数运算更稳定在 fp32；之后会再 cast 回去。
* 得到 `x_complex` 形状为 `[..., seq_len, d_k/2]`（复数张量）。

```python
freqs_cis = self.freqs_cis[token_positions]
```

* 用 `token_positions` 从 `[max_seq_len, d_k/2]` 里取出每个 token 的旋转因子，得到 `[..., seq_len, d_k/2]` 的复数张量。

```python
if x_complex.dim() == 4: # 典型的 (batch, head, seq, dim)
    freqs_cis = freqs_cis.unsqueeze(1)
```

* 常见形状 `(B, H, T, d_k/2)`：`freqs_cis` 目前是 `(B, T, d_k/2)`（没有 head 维），在第 1 维 `unsqueeze` 一下，变 `(B, 1, T, d_k/2)`，以便在所有 **heads** 上广播同样的旋转（每个 head 用相同的位置信息）。

```python
x_rotated = x_complex * freqs_cis
```

* 复数逐元素相乘，即每个二维平面上的旋转：$z' = z \cdot e^{j\theta}$。

```python
x_out = torch.view_as_real(x_rotated)
x_out = x_out.reshape(*x.shape)
return x_out.type_as(x)
```

* 把复数张量还原回实数的最后两维 `(real, imag)`，再 `reshape` 回原始形状 `... d_k`。
* 最后 cast 回输入 `x` 的 dtype（比如 fp16/bf16）。

In [32]:
    def forward(
        self, 
        x: Float[Tensor, "... seq_len d_k"], 
        token_positions: Int[Tensor, "... seq_len"]
    ) -> Float[Tensor, "... seq_len d_k"]:
        x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
        freqs_cis = self.freqs_cis[token_positions]
        if x_complex.dim() == 4: # 典型的 (batch, head, seq, dim)
            freqs_cis = freqs_cis.unsqueeze(1)
        x_rotated = x_complex * freqs_cis
        x_out = torch.view_as_real(x_rotated)
        x_out = x_out.reshape(*x.shape)
        return x_out.type_as(x)

## 9.4 RMSNorm

1. 首先有个问题：**为什么需要归一化？**

    1. 稳定梯度，防止梯度爆炸/消失
    
        * 神经网络很深时，激活和梯度会在传播中不断放大或缩小。
        * 没有归一化 → 梯度可能 **爆炸**（过大）或 **消失**（趋近 0）。
        * 归一化让输入分布保持在一个**适中的尺度**，梯度更新更稳定。
    
    2. 减少“内部协变量偏移” (Internal Covariate Shift)
    
        * 每一层的输入分布会随着训练不断变化。
        * 如果分布漂移太大，下一层需要不停适应 → 训练慢且不稳。
        * 归一化保证每层的输入分布更“平滑、稳定”，相当于提供一个统一的参考系。
    
    3. 加快收敛
    
        * 数据分布规范后，优化器（SGD, Adam 等）能更高效地更新权重。
        * 实验上：有归一化的模型，往往能在更大学习率下收敛，更快达到较优解。
    
    4. 提高数值稳定性
    
        * 在低精度训练（FP16, BF16）下，数值范围过大会导致溢出/下溢。
        * 归一化把值控制在合理范围，防止数值不稳定。
    
    5. 改善泛化性能
    
        * 归一化会引入一些“噪声效应”（尤其是 batchnorm），对模型有正则化作用。
        * 可以减少过拟合，让模型对新数据更鲁棒。
    
    总结：**归一化是为了让每一层的输入保持在一个稳定、可控的尺度上 → 梯度好算、训练快、收敛稳、泛化好。**

2. 什么是RMSNorm？（和LayerNorm对比着看）

    RMSNorm（Root Mean Square Layer Normalization）是一种**按最后一维做归一化**的层，和 LN 一样是“token-wise”的，但**它不做去均值**，只用向量的均方根（RMS）来缩放，然后乘一个可学习的逐维缩放参数 `γ`。常见实现 **没有偏置 `β`** 。
    
    * 目标直觉：把每个向量投影到“定长”的球面上（RMS 归一），但不改变它的均值结构（不中心化）。
    
    1. 数学定义
    
    设输入向量为 $x\in\mathbb{R}^d$（通常是某个 token 的隐藏状态最后一维），$\epsilon>0$ 为数值稳定项，$\gamma\in\mathbb{R}^d$ 为可学习缩放参数。
    
    **LayerNorm（LN）**
    
    $$
    \mu=\frac{1}{d}\sum_{i=1}^d x_i,\quad
    \sigma^2=\frac{1}{d}\sum_{i=1}^d (x_i-\mu)^2
    $$
    
    $$
    \hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}},\quad
    y_i=\gamma_i\hat{x}_i+\beta_i
    $$
    
    **RMSNorm**
    
    $$
    \mathrm{rms}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^d x_i^2}
    $$
    
    $$
    \hat{x}_i=\frac{x_i}{\sqrt{\mathrm{rms}(x)^2+\epsilon}}
    =\frac{x_i}{\sqrt{\frac{1}{d}\sum_{j=1}^d x_j^2+\epsilon}},\quad
    y_i=\gamma_i\hat{x}_i
    $$
    
    > 可以把分母理解为 $\|x\|_2/\sqrt{d}$（加了 $\epsilon$）。
    
    2. 为什么近几年大模型中都是使用 RMSNorm 了？
    
        1. **计算更少**：LN 需要计算均值和方差，RMSNorm 只要**平方和一次归约**即可；同时相比于 LN,RMSNorm 还少加了一个偏置项，虽然加法等在计算中占比很少，但是其内存调转引起的时延占比非常非常高；
        2. **反向传播简单**：没有取均值与方差项，梯度表达式更简单，数值更稳定（尤其混合精度、低比特场景）。
        3. **能够保留均值（偏移）信息**：LN 强制零均值，RMSNorm 不会抹去均值方向上的信息；实践上，在**预归一化（Pre-Norm）Transformer**的残差通路里，这常常配合得很好（许多现代解码式 LLM——如 LLaMA 系列、Mistral 等——采用 RMSNorm）。
    
    > 这里是标准答案，是不是很抽象？可以看后面的一个板块，一个很简单的例子就能很形象的解释清楚。

这里我们从非直觉的角度讲讲 LayerNorm 和 RMSNorm 到底有什么区别？（老实说，我看了半天这俩玩意的数学表达式，没有一点数学直觉，可能是我天生比较愚钝）

我们进行一个简单的例子，假设有三句话的 embedding：

```
句子 A: [1000, 1005, 995]
句子 B: [0.9, 1.0, 1.1]
句子 C: [-2, 0, 2]
```

现在我们来对他们进行一个简单的计算：

In [33]:
import torch
import torch.nn as nn
import numpy as np

vectors = {
    "A": torch.tensor([1000.0, 1005.0, 995.0]),
    "B": torch.tensor([0.9, 1.0, 1.1]),
    "C": torch.tensor([-2.0, 0.0, 2.0])
}

layer_norm = nn.LayerNorm(3, eps=1e-5)

def rms_norm(x, eps=1e-5):
    rms = torch.sqrt(torch.mean(x**2))
    return x / (rms + eps)

for name, vector in vectors.items():
    ln_result = layer_norm(vector)
    rms_result = rms_norm(vector)
    
    print(f"\n向量 {name}:")
    print(f"原始向量: {vector}")
    print(f"LayerNorm: {ln_result}")
    print(f"RMSNorm:   {rms_result}")


向量 A:
原始向量: tensor([1000., 1005.,  995.])
LayerNorm: tensor([ 0.0000,  1.2247, -1.2247], grad_fn=<NativeLayerNormBackward0>)
RMSNorm:   tensor([1.0000, 1.0050, 0.9950])

向量 B:
原始向量: tensor([0.9000, 1.0000, 1.1000])
LayerNorm: tensor([-1.2238,  0.0000,  1.2238], grad_fn=<NativeLayerNormBackward0>)
RMSNorm:   tensor([0.8970, 0.9967, 1.0963])

向量 C:
原始向量: tensor([-2.,  0.,  2.])
LayerNorm: tensor([-1.2247,  0.0000,  1.2247], grad_fn=<NativeLayerNormBackward0>)
RMSNorm:   tensor([-1.2247,  0.0000,  1.2247])


这里就能很明显的观察到

1. **A vs B**

   * 输入分布差异：只是整体尺度不同（A 数值 \~1000，B 数值 \~1）。
   * **LN 结果差异很大**（A 归一化后 = `[0, 1.22, -1.22]`，B = `[-1.22, 0, 1.22]`）。
   * **RMSNorm 结果差不多**（A ≈ `[1, 1, 1]`，B ≈ `[0.9, 1.0, 1.1]`）。

2. **B vs C**

   * 输入分布差异：B = `[0.9, 1.0, 1.1]`（小波动，偏正），C = `[-2, 0, 2]`（对称，跨度大）。
   * **LN 结果几乎一样**（都标准化到 `[-1.22, 0, 1.22]`）。
   * **RMSNorm 结果明显不同**（B ≈ `[0.9,1.0,1.1]`，C ≈ `[-1.22, 0, 1.22]`）。

这说明了什么？

1. **LN 强行拉正分布**

    * LN 一定要“零均值 + 单位方差”。
    * 它忽略了向量的整体偏移和尺度，只保留**相对差值的形状**。
    * 所以 B（小波动）和 C（大跨度）被“拉平”，变得一模一样。
    * 对模型来说，**分布差异信息丢失**。

    **在大模型训练里**：
    这意味着 LN 会抹掉一些 embedding 或激活中的**全局偏移信息**（例如某个 token embedding 的平均值里可能包含语义或位置信息），模型需要额外参数去“重新学回”这些信息。

2. **RMSNorm 保留了均值和形状差异**

    * RMSNorm 只保证“长度统一”，但不改变均值。
    * 所以：
    
      * A vs B：数值尺度不同，但方向相似 → RMSNorm 后结果差不多。
      * B vs C：分布形状差异大（对称 vs 偏正） → RMSNorm 后仍能看出来。
    * 换句话说，RMSNorm **保留了分布的“几何特征”**，而 LN 抹平了它。
    
    **在大模型训练里**：
    
    * 大模型输入（embedding）往往带有**语义偏移**（例如语法模式、上下文趋势）。
    * RMSNorm 不会去掉这些信息，只是统一尺度，让优化更稳。
    * LN 则会过度干预 → 破坏了本来有用的统计特征。

3. **大模型更偏好 RMSNorm 的原因**

    * **保留信息**：
      RMSNorm 不会强制零均值，embedding/hidden states 的“漂移”信息被保留，对语义建模有帮助。
    * **数值稳定**：
      LN 要算均值+方差，尤其是小波动（像 B）时，数值容易被放大（分母小 → 输出大），导致训练不稳；RMSNorm 没这个问题。
    * **计算效率**：
      RMSNorm 只需一次平方和规约，计算更快。
    
    这就是为什么 **T5、LLaMA、PaLM** 等大模型，几乎全都放弃 LN，改用 RMSNorm。

搞明白了 RMSNorm 的原理，实现当然非常简单。

### 9.4.1 __init__

```python
self.d_model = d_model
```
因为 RMSNorm 中有一个可以学习的缩放参数，每个维度都需要一个独立的缩放因子，大小就是 `d_model`.

```python
elf.eps = eps
```
数值稳定项，防止分母为 0 或太小，RMS 公式里会用到，一般默认 1e-5。

```python
self.weight = nn.Parameter(torch.ones(d_model, **factory_kwargs))
```
这是 RMSNorm 的**逐维缩放参数** `γ`（gain）,形状 `(d_model,)`，默认初始化为 1，表示“初始不改变每个通道的比例”。

In [34]:
class RMSNorm(nn.Module):
    def __init__(
        self,
        d_model: int,
        eps: float = 1e-5,
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        factory_kwargs = {"device": device, "dtype": dtype}
        self.weight = nn.Parameter(torch.ones(d_model, **factory_kwargs)x

### 9.4.2 forward

```python
in_dtype = x.dtype
x = x.to(torch.float32)
```
保留原始精度，再统一到 fp32 计算。

```python
x.pow(2).mean(-1, keepdim=True)
```
`x.pow(2)`原地平方，`mean(-1, keepdim=True)`沿着最后一个维度求均值，同时保留该维度。

```python
torch.rsqrt()
```
求平方根的倒数。

In [35]:
    def forward(self, x: Float[Tensor, "... d_model"]) -> Float[Tensor, "... d_model"]:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim = True) + self.eps)
        normalized_x = x * rms * self.weight
        return normalized_x.to(in_dtype)

## 9.5 feed_forward

标准的 transformer 架构中 FFN（Feed-Forward Network） 是 MLP + 激活函数 + MLP 的实现，但是现在都用 SwiGLU / GEGLU 的实现直接代替 FFN 的实现了（因为效果更好）。

所以本部分 FFN 的实现就是一个 SwiGLU。

### 9.5.1 SwiGLU

1. **数学原理（SwiGLU 是什么？）**

FFN 的“**门控线性单元**之前都是使用经典 GLU（Dauphin et al., 2017），定义为：

$$
\mathrm{GLU}(x)=\big(W_v x + b_v\big)\;\odot\;\sigma\!\big(W_g x + b_g\big)
$$

其中 $\sigma$ 是 Sigmoid，$\odot$ 是逐元素乘法（Hadamard 积）。直觉上讲是这样的：一条支路产生“值”（value），另一条支路产生“门”（gate），用门来按元素选择/抑制值。

**SwiGLU**（Shazeer, 2020 的 GLU variants）把门控激活从 Sigmoid 换成 **SiLU/Swish**，经验上在大模型里更好：

$$
\mathrm{SwiGLU}(x)=\big(W_v x + b_v\big)\;\odot\;\mathrm{SiLU}\!\big(W_g x + b_g\big),
\quad \mathrm{SiLU}(z)= z\cdot\sigma(z)
$$

展开后：

$$
\mathrm{SwiGLU}(x) = (W_v x + b_v) \odot \Big[(W_g x + b_g) \cdot \sigma(W_g x + b_g)\Big]
$$

也就是：

$$
\mathrm{SwiGLU}(x)
= (W_v x + b_v) \odot (W_g x + b_g) \odot \sigma(W_g x + b_g)
$$

在 Transformer 里通常还会再接一个回投影，把维度投回 $d_{\text{model}}$：

$$
\mathrm{FFN}(x)= W_o^\top\Big(\mathrm{SwiGLU}(x)\Big) + b_o
$$

这就对应“三个线性层 + 一个逐元素乘”的结构：两条并行输入投影（产生 value 与 gate），逐元素相乘，最后一个输出投影。


2. **为什么一般 $d_{\text{ff}} \approx \frac{8}{3}d_{\text{model}}$ ？**

传统 GELU-MLP 用 $d_{\text{ff}}=4d$ 且只有两次投影（$d\to4d\to d$），参数量约 $8d^2$。
SwiGLU 有 **三** 次投影（两入一出）：参数量 $\approx 3 d\cdot d_{\text{ff}}$。
为了和传统 MLP 的参数/算力预算相当，令 $3 d\cdot d_{\text{ff}}\approx 8d^2\Rightarrow d_{\text{ff}}\approx \frac{8}{3}d$。

3. **为什么 $d_{\text{ff}}$ 要对齐到 64 的倍数？**

GPU 上的矩阵乘法有个硬件优化要求：

- 如果矩阵维度能整除某些粒度（一般是 8、16、32、64），就能充分利用 Tensor Core 或 SIMD 单元。
- 否则就要填充（padding），导致性能下降。
```python
d_ff = int((8/3) * d_model)
d_ff = (d_ff + 63) // 64 * 64
```
这是一个向上取64整数倍的技巧。

### 9.5.2 __init__

d_model：Transformer 各层之间传递的隐藏向量维度    
d_ff：FFN 内部的中间隐藏层维度

SwiGLU 的核心公式：

$$
\mathrm{SwiGLU}(x) = \big(W_v x + b_v\big)\;\odot\;\mathrm{SiLU}(W_g x + b_g)
$$

接着再回投影：

$$
\mathrm{FFN}(x) = W_o^\top \Big(\mathrm{SwiGLU}(x)\Big) + b_o
$$

与代码中的 w123 的对应关系分别为：

* **`w1`** ⟶ $W_g$：
  产生 gate 分支：$\mathrm{SiLU}(W_g x + b_g)$。

* **`w2`** ⟶ $W_v$：
  产生 value 分支：$(W_v x + b_v)$。

* **`w3`** ⟶ $W_o$：
  输出投影，把逐元素乘的结果从 $d_{\text{ff}}$ 投回 $d_{\text{model}}$。

In [36]:
class SwiGLU(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int | None = None,
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        if d_ff is None:
            d_ff = int((8/3) * d_model)
            d_ff = (d_ff + 63) // 64 * 64
        self.w1 = Linear(d_model, d_ff, **factory_kwargs)  # 使用我们自己实现的 Linear
        self.w2 = Linear(d_model, d_ff, **factory_kwargs)
        self.w3 = Linear(d_ff, d_model, **factory_kwargs)

### 9.5.3 forward

没什么好讲的，按照公式来就行。

In [37]:
    def forward(
        self,
        x: Float[Tensor, ".. d_model"]
    )-> Float[Tensor, "... d_model"]:
        # FFN(x) = W3(SiLU(W1x) ⊙ W2x)
        gate = self.w1(x)
        hidden = silu(self.w2(x))        # 使用我们自己实现的 silu
        return self.w3(hidden * gate)

## 9.6 attention

我们分两部分：
1. scaled_dot_product_attention
2. MultiHeadSelfAttention

分别实现。
注意力机制这里就不讲了，感兴趣的可以移步 [Transformer](https://github.com/CliffKai/DeepL-by-Pytorch/blob/master/Paper%20Reading/NLP/Transformer.md) 细看。

### 9.6.1 scaled_dot_product_attention

#### 9.6.1.1 关于 mask

1. **没有 mask 的注意力**

缩放点积注意力的分数是：

$$
\text{scores} = \frac{QK^T}{\sqrt{d_k}}
$$

这是一个形状为 $(\text{queries}, \text{keys})$ 的矩阵。
每个 `query` 向量都会和所有 `key` 做点积，衡量相似度，然后用 softmax 归一化，得到注意力权重。

这样，**任意一个 query 都能看到整个序列的所有位置**。

2. **为何要引入 mask？在实际任务中，有时我们并不希望 query 能看到所有 key：**
    1. **Padding mask**
        * NLP 训练时，句子长度不同，短的句子会用 `<PAD>` 填充。
        * 这些 `<PAD>` token 不携带信息，如果不屏蔽，它们会干扰注意力分布。
        * `mask` 在这些位置上填 `False`，把 `scores` 设置成 $-\infty$，softmax 后权重为 0，相当于完全忽略。
    2. **Causal mask (因果掩码)**
        * 在语言模型训练时，生成第 $t$ 个词时，只能看见位置 $[0,1,...,t]$，不能偷看未来的 token。
        * 因果 mask 就是一个 **上三角矩阵**（未来位置被屏蔽）。
        * 这样可以保证自回归（autoregressive）生成符合因果性。
    3. **任意掩码（任务相关）**
        * 在一些特殊任务（图结构、跨模态）中，我们可能只允许某些位置之间互相关注。
        * 这时 mask 就是一个自定义的布尔矩阵。

3. **代码里的作用**

```python
if mask is not None:
    scores = scores.masked_fill(mask == False, float("-inf"))
```

* `mask` 形状是 `(..., queries, keys)`，和 `scores` 对应。
* 在 mask 中为 `False` 的地方，直接把分数设为 $-\infty$。
* softmax 时，$\exp(-\infty) = 0$，所以这些位置的注意力权重完全消失。

4. **举个栗子**

比如输入序列长度为 4，因果 mask 是：

$$
\begin{bmatrix}
1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 \\
1 & 1 & 1 & 0 \\
1 & 1 & 1 & 1 \\
\end{bmatrix}
$$

* 第 0 个 token 只能看自己；
* 第 2 个 token 可以看 \[0,1,2]，不能看未来的 \[3]；

**总结**：    
`mask` 的本质作用就是 **控制信息流动**，通过在 softmax 前把非法位置的分数设为 $-\infty$，确保注意力机制只在允许的范围内进行。

#### 9.6.1.2 关于 $ {\sqrt{d_k}} $

$$
\text{score}=\frac{q\cdot k}{\sqrt{d_k}}
$$

首先说核心目的：是为了**让分布的尺度与维度无关**，避免 softmax 在大维度上过于**饱和**，从而**保持稳定的梯度与可训练性**。

下面我将从下面几个角度进行解释（也是我自己尝试去理解的几个角度）：

1. **统计视角：方差归一化的推导**

把 $q=(q_1,\dots,q_{d_k})$、$k=(k_1,\dots,k_{d_k})$ 看成零均值、独立同分布、方差为 $\sigma^2$ 的随机向量（这是常见的初始化/规范化后的合理近似）：

* 点积：$\;q\cdot k=\sum_{i=1}^{d_k} q_i k_i$.
* 因为 $q_i,k_i$ 独立且 $\mathbb{E}[q_i k_i]=0$，有

  $$
  \mathrm{Var}(q\cdot k)
  \;=\;\sum_{i=1}^{d_k}\mathrm{Var}(q_i k_i)
  \;=\;\sum_{i=1}^{d_k}\mathbb{E}[q_i^2]\mathbb{E}[k_i^2]
  \;=\;d_k\,\sigma^4.
  $$

因此未缩放的打分的**标准差**是 $\sqrt{d_k}\,\sigma^2$，会**随维度 $\sqrt{d_k}$** 增大。

把分子除以 $\sqrt{d_k}$ 后：

$$
\mathrm{Var}\!\left(\frac{q\cdot k}{\sqrt{d_k}}\right)=\frac{d_k\,\sigma^4}{d_k}=\sigma^4,
$$

其尺度**与维度无关**（若再考虑把 $\sigma$ 通过初始化/归一化调到 1，方差就稳定在常数级）。

2. **对 softmax 与梯度的影响**

softmax 对输入尺度很敏感。若 logits 的标准差随 $\sqrt{d_k}$ 变大：

* softmax 会越来越接近 one-hot 分布（某个键几乎为 1，其余近 0）；
* 此时梯度 $\nabla \text{softmax}$ 变小（饱和），学习变慢甚至不稳定。

把 logits 统一缩放为 $(q\cdot k)/\sqrt{d_k}$ 后，不论头的维度多大，进入 softmax 的数值落在**相近的可训练范围**，既不 one-hot、也不太平，**梯度健康**、训练更稳。

In [ ]:
def scaled_dot_product_attention(
    q: Float[Tensor, "... quaries d_k"],
    k: Float[Tensor, "... keys d_k"],
    v: Float[Tensor, "... values d_v"],
    mask: Bool[Tensor, "... quaries keys"] | None = None,
) -> Float[Tensor, "... quaries d_v"]:
    d_k= q.size(-1)
    scores = einsum(q, k, "... q d, ...k d -> ... q k") / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == False, float("-inf"))
    atten_weights = softmax(scores, dim=-1)         # 这里 softmax 也用我们之前自己的实现
    output = einsum(atten_weights, v, "... q k, ... v d -> ... q d")
    return output

> 这里必须得强烈推荐一波 einops.einsum，这玩意用起来太舒服了

### 9.6.2 MultiHeadSelfAttention

#### 9.6.2.1 __init__

d_model：模型的隐藏维度      
num_heads：注意力头的数量         
rope：我们在 9.3 部分实现的旋转位置编码         
```python
if d_model % num_heads != 0:
    raise ValueError("d_model must be divisible by num_heads")
self.d_head = d_model // num_heads
```
多头注意力把隐藏维 d_model 均分给 num_heads 个头，每头的头维是 d_head。后面会把最后一维 reshape 成 (h, d_head)，这样实现最高效。

```python
self.q_proj = Linear(d_model, d_model)
self.k_proj = Linear(d_model, d_model)
self.v_proj = Linear(d_model, d_model)
self.output_proj = Linear(d_model, d_model)
```
典型做法：单个全连接从 `d_model -> d_model`，随后再用 `einops.rearrange` 把最后一维拆成 `(h, d_head)`。
数学上，对输入 `x ∈ ℝ^{B×S×d_model}`：

$$
Q = x W_Q,\quad K = x W_K,\quad V = x W_V,\quad W_Q,W_K,W_V∈ℝ^{d_{model}\times d_{model}}
$$

输出合并后再乘 $W_O ∈ ℝ^{d_{model}\times d_{model}}$。

```python
self.register_buffer("causal_mask", None, persistent=False)
```
让因果掩码随设备迁移，但是不优化不写入模型参数。

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        rope: RotaryPositionalEmbedding,       # 这里使用我们自己实现的 Rope
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.rope = rope
        factory_kwargs = {"device": device, "dtype": dtype}

        self.q_proj = Linear(d_model, d_model, **factory_kwargs)
        self.k_proj = Linear(d_model, d_model, **factory_kwargs)
        self.v_proj = Linear(d_model, d_model, **factory_kwargs)
        self.output_proj = Linear(d_model, d_model, **factory_kwargs)

        self.register_buffer("casual_mask", None, persistent=False)

#### 9.6.2.2 get_causal_mask

**先讲讲torch.triu()**    
`torch.triu(input, diagonal=0, *, out=None)` 构造并返回与 `input` 形状相同的张量，但只保留“上三角”部分，其余元素置零（对 `bool` 张量则置为 `False`）。
适用于矩阵或形状为 `(..., m, n)` 的批量矩阵。

**精确定义**（对最后两维的元素）：

$$
y[\dots,i,j] =
\begin{cases}
x[\dots,i,j], & \text{若 } j - i \ge \texttt{diagonal} \\
0, & \text{否则}
\end{cases}
$$

* `diagonal = 0`：保留下**主对角线及其上方**。
* `diagonal > 0`：向上偏移，保留更“靠上”的对角线（更少元素）。
* `diagonal < 0`：向下偏移，额外保留部分下三角（更多元素）。

还有原地版本 `torch.triu_(input, diagonal=0)`。

**举个栗子**

```python
import torch

A = torch.tensor([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [9,10,11,12]], dtype=torch.float)

torch.triu(A, diagonal=0)
# tensor([[ 1.,  2.,  3.,  4.],
#         [ 0.,  6.,  7.,  8.],
#         [ 0.,  0., 11., 12.]])

torch.triu(A, diagonal=1)
# 主对角线上方开始保留（主对角也被清零）
# tensor([[ 0.,  2.,  3.,  4.],
#         [ 0.,  0.,  7.,  8.],
#         [ 0.,  0.,  0., 12.]])

torch.triu(A, diagonal=2)
# 主对角线上方开始保留（主对角也被清零）
# tensor([[ 0.,  0.,  3.,  4.],
#         [ 0.,  0.,  0.,  8.],
#         [ 0.,  0.,  0.,  0.]])

torch.triu(A, diagonal=-1)
# 比主对角线再“放宽”一条下对角
# tensor([[ 1.,  2.,  3.,  4.],
#         [ 5.,  6.,  7.,  8.],
#         [ 0., 10., 11., 12.]])
```

**批量矩阵也可以：**

```python
B = torch.arange(2*3*3).view(2,3,3)   # 形状 [2, 3, 3]
torch.triu(B)  # 对两个 3x3 矩阵分别取上三角
```

**与 causal mask 的关系**

在自回归注意力里，我们通常要**屏蔽“看未来”的位置**。长度为 `L` 的序列常用：

```python
L = 5
# True 表示“在上三角（未来位置）”
future = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
# masked_fill 里 True 位置会被屏蔽（置 -inf）
scores = scores.masked_fill(future, float("-inf"))
```

**常见用法**

1. **构造布尔上三角掩码：**

```python
L = 128
tri_mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device="cuda"), diagonal=0)
```

2. **对任意张量“清掉”下三角：**

```python
X = torch.randn(3, 4, device="cuda")
upper = torch.triu(X)  # 下三角元素被置 0
```

3. **原地版本（慎用在需要梯度的中间结果上）：**

```python
X.triu_()  # 直接把 X 的下三角清零
```

**再来看代码**

```python
if self.causal_mask is None or self.causal_mask.size(0) < seq_len:
```
第一次使用 causal_mask 或者之前的 causal_mask 尺寸不够，就重新分配一个。

```python
mask = torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)
self.causal_mask = ~mask
```
前一句就不用说了，后一句中 `~` 是每一位都取反的意思，上三角变成了下三角。

In [ ]:
    def get_causal_mask(
        self,
        seq_len: int,
        device: torch.device,
    ) -> Bool[Tensor, "seq_len, seq_len"]:
        if self.causal_mask is None or self.causal_mask.size(0) < seq_len:
            mask = torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)
            self.causal_mask = ~mask
        return self.causal_mask[:seq_len, :seq_len]

#### 9.6.2.3 forward

1. **线性投影：得到 Q/K/V**

```python
q = self.q_proj(x)
k = self.k_proj(x)
v = self.v_proj(x)
```

2. **拆分为多头**

```python
q = rearrange(q, "b s (h d) -> b h s d", h=self.num_heads)
k = rearrange(k, "b s (h d) -> b h s d", h=self.num_heads)
v = rearrange(v, "b s (h d) -> b h s d", h=self.num_heads)
```

* 把最后一维 `D` 按 `H × Dh` 重排成 `[B, H, S, Dh]`。
* 为什么：多头注意力需要并行地在 `H` 个独立子空间里做 SDPA（scaled dot-product attention）。
* 形状变化：

  * 之前：`[B, S, D]`
  * 之后：`[B, H, S, Dh]`，其中 `Dh = D/H`。

> 只 reshape 最后一维，是因为“头”的维度就埋在最后一维里（把一个大向量拼成 H 个小向量），`B`、`S` 两维的逻辑不变。

3. **应用 RoPE（旋转位置编码）**

```python
q = self.rope(q, token_positions)
k = self.rope(k, token_positions)
```

看 9.3 RoPE 部分。

4. **因果掩码（只看自己与过去）**

```python
causal_mask = self.get_causal_mask(seq_len, x.device)
```

* `get_causal_mask` 生成（或复用缓存的）下三角布尔矩阵，形状 `[S, S]`：

  * `mask[i, j] = True` 表示 **第 i 个查询**可以看 **第 j 个键**；
  * 只允许 `j ≤ i`（含主对角线），所以是**因果**（当前只能看过去和自己）。
* 为什么：自回归语言模型需要防止“偷看”未来 token。
* 额外点：把这个掩码注册成 buffer 缓存起来，避免每次前向都重新分配（节省时间和显存）。

> 传入 SDPA 时，`[S, S]` 会**自动广播**到 `[B, H, S, S]`（对所有 batch、所有头相同）。

5. **缩放点积注意力（SDPA）**

```python
attn_output = scaled_dot_product_attention(q, k, v, mask=causal_mask)
```

看 9.6.1 scaled_dot_product_attention 部分。

6. **合并多头**

```python
attn_output = rearrange(attn_output, "b h s d -> b s (h d)")
```

* 做了什么：把 `H` 个头的输出在通道维度拼回去（逆步骤 2）。
* 形状变化：`[B, H, S, Dh] → [B, S, D]`。

7. **输出投影**

```python
return self.output_proj(attn_output)
```

* 做了什么：最后一个线性层把拼好的多头输出再映射到 `D` 维（形状保持 `[B, S, D]`）。

In [ ]:
    def forward(
        self,
        x: Float[Tensor, "batch seq_len d_model"],
        token_positions: Int[Tensor, "batch seq_len"],
    ) -> Float[Tensor, "batch seq_len d_model"]:
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = rearrange(q, "b s (h d) -> b h s d", h = self.num_heads)
        k = rearrange(k, "b s (h d) -> b h s d", h = self.num_heads)
        v = rearrange(v, "b s (h d) -> b h s d", h = self.num_heads)
        q = self.rope(q, token_positions)
        k = self.rope(k, token_positions)
        causal_mask = self.get_causal_mask(seq_len=x.size(1), device=x.device)
        attn_output = scaled_dot_product_attention(q, k, v, mask=causal_mask)
        attn_output = rearange(attn_output, "b h s d -> b s (h d)")
        return self.output_proj(attn_output)

## 9.7 transformer

接下来就是用我们自己实现的各个模块来构造一个 transformer block，然后在组成一个 transformer 模型，这一部分需要实现：
1. TransformerBlock
2. TransformerLM

实现规范见下图 Figure 1 与 Figure 2：           

![Figure 1](images/Figure_1_An_overview_of_our_Transformer_language.png)

![Figure 2](images/Figure_2_A_pre-norm_Transformer_block.png)

### 9.7.1 TransformerBlock

#### 9.7.1.1 __init__

构建 TransformerBlock 中所需要的各个模块。

In [42]:
class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int,
        rope: RotaryPositionalEmbedding,
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.ln1 = RMSNorm(d_model, **factory_kwargs)
        self.attn = MultiHeadSelfAttention(d_model, num_heads, rope, **factory_kwargs)
        self.ln2 = RMSNorm(d_model, **factory_kwargs)
        self.ffn = SwiGLU(d_model, d_ff, **factory_kwargs)

#### 9.7.1.2 forward

按照 Figure 2 构建前向过程即可。

In [43]:
    def forward(
        self,
        x: Float[Tensor, "batch seq_len d_model"],
        token_positions: Int[Tensor, "batch seq_len"],
    ) -> Float[Tensor, "batch seq_len d_model"]:
        residual = x
        x_norm = self.ln1(x)
        attn_out = self.attn(x_norm, token_position)
        x = residual + attn+out

        residual = x
        x_norm = self.ln2(x)
        ffn_out = self.ffn(x_norm)
        x = residual + ffn_out

        return x

### 9.7.2 TransformerLM

#### 9.7.2.1 __init__

构建 TransformerLM 中所需要的各个模块。

* `vocab_size`：词表大小，输出层的维度也是这个。    
* `context_length`：最大序列长度，注意力与RoPE的缓存上界。    
* `rope_theta`：RoPE 的基频（频率尺度）。θ 越大，角频率越小（周期更长），对**长度外推**更友好但短程分辨率略降；反之亦然。

In [45]:
class TransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        context_length: int,
        d_model: int,
        num_layer: int,
        num_heads: int,
        d_ff: int,
        rope_theta: float,
        device: torch.device | str | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        d_head = d_model // num_heads
        self.token_embedding = Embedding(vocab_size, d_model, **factory_kwargs)
        rope = RotaryPositionalEmbedding(d_head, context_length, rope_theta, device=device)
        
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, rope, **factory_kwargs)
            for _ in range(num_layers)
        ])

        self.ln_final = RMSNorm(d_model, **factory_kwargs)
        self.lm_head = Linear(d_model, vocab_size, **factory_kwargs)

#### 9.7.2.2 forward

按照 Figure 1 构建前向过程即可。

* `in_indices`: Token id 形状是`[B, L]`

In [46]:
    def forward(
        self,
        in_indices: Int[Tensor, "batch seq_len"]
    ) -> Float[Tensor, "batch seq_len vocab_size"]:
        batch_size, seq_len = in_indices.shape
        device = in_indices.device

        token_positions = torch.arrange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)

        x = self.token_embeddings(in_indeces)
        for layer in self.layer:
            x = layer(x, token_positions)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits